[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/01_python_for_ml/07_common_python_errors.ipynb)

# Bài 07 — Những lỗi thường gặp trong Python (và cách sửa)

**🎯 Mục tiêu**
- Đọc traceback nhanh: biết lỗi gì, ở dòng nào của **code mình**.
- Nhận diện ~12 loại lỗi (exception) gặp nhiều nhất khi mới học Python/ML, hiểu nguyên nhân và cách sửa.
- Nhận diện **lỗi logic** — code chạy, không báo lỗi, nhưng kết quả sai.
- Xử lý lỗi đúng cách: `try/except` hẹp, `raise` với thông điệp rõ ràng, `assert` cho giả định nội bộ.

> Bài này hiện thực hoá mục *"Những lỗi thường gặp trong Python"* trong kế hoạch tuần 1 của khoá AIVN (bản gốc chỉ có liên kết ngoài). [Bài 06](06_debugging_shapes_and_performance.ipynb) đã nói về lỗi shape/NaN đặc thù của ML; bài này là các lỗi **Python thuần** — gặp hằng ngày ở mọi chương sau.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/01_python_for_ml

In [ ]:
import math
import traceback
import warnings

import numpy as np


def show_error(code: str) -> None:
    """Chạy một đoạn code mẫu (trên bản sao biến của notebook); nếu có lỗi thì in `LoạiLỗi: thông điệp`."""
    try:
        exec(compile(code, "<ví dụ>", "exec"), dict(globals()))
        print("✅ không có lỗi")
    except Exception as e:  # noqa: BLE001 — cố ý bắt mọi lỗi để minh hoạ
        print(f"❌ {type(e).__name__}: {e}")


show_error("x = 1 + 1")
show_error("x = 1 / 0")

## 1. Đọc traceback trong 30 giây

Khi có lỗi, Python in **traceback** — chuỗi lời gọi hàm dẫn tới lỗi. Đọc **từ dưới lên**:

1. **Dòng cuối**: loại lỗi + thông điệp — thường đã nói gần hết vấn đề.
2. **Đi ngược lên**: tìm dòng đầu tiên thuộc **code của bạn** (không phải trong thư viện) — đó là nơi cần sửa.
3. Các dòng phía trên nữa cho biết *ai đã gọi* tới đó.

In [ ]:
def normalize(row):
    total = sum(row)
    return [v / total for v in row]


def preprocess(rows):
    return [normalize(r) for r in rows]


try:
    preprocess([[1, 2, 3], [0, 0, 0]])
except ZeroDivisionError:
    print(traceback.format_exc())

Đọc từ dưới lên: `ZeroDivisionError: division by zero` → xảy ra trong `normalize` (dòng `v / total`) → được gọi từ `preprocess`. Nguyên nhân: hàng `[0, 0, 0]` có tổng bằng 0. Cách sửa phụ thuộc **ý nghĩa** dữ liệu: bỏ hàng đó, hoặc quy ước trả về toàn 0, hoặc báo lỗi rõ ràng hơn (mục 5).

> 💡 Trong Jupyter, traceback đầy đủ còn tô màu và (Python ≥ 3.10) gợi ý kiểu *"Did you mean: 'total'?"* khi bạn gõ sai tên biến.

## 2. Lỗi cú pháp: `SyntaxError`, `IndentationError`

Lỗi cú pháp được phát hiện **trước khi chạy** — cả cell không chạy dòng nào. Python ≥ 3.10 có thông điệp khá thân thiện:

In [ ]:
show_error("x = 5\nif x > 0\n    print(x)")              # thiếu dấu ':'
show_error("print('xin chào'")                           # thiếu dấu ')'
show_error("x = 1\nif x = 1:\n    print(x)")              # '=' (gán) thay vì '==' (so sánh)
show_error("def f():\nreturn 1")                          # thân hàm không thụt lề
show_error("x = 1\n    y = 2")                            # thụt lề thừa

| Thông điệp | Nguyên nhân hay gặp |
|---|---|
| `expected ':'` | thiếu `:` sau `if`, `for`, `def`, `class`, `while`, `else` |
| `'(' was never closed` | thiếu ngoặc đóng — lỗi thường được báo ở **dòng sau** dòng thật sự sai |
| `invalid syntax. Maybe you meant '==' …` | dùng `=` trong điều kiện |
| `expected an indented block` / `unexpected indent` | thụt lề sai; trộn tab và dấu cách |

> ✅ Để editor (VS Code, PyCharm, Jupyter) tự thụt lề 4 dấu cách; không trộn tab với dấu cách.

## 3. Mười lỗi runtime gặp nhiều nhất

### 3.1 `NameError` — dùng tên chưa được định nghĩa

In [ ]:
show_error("total = 10\nprint(totl)")          # gõ sai tên
show_error("print(accuracy)")                  # trong notebook: quên chạy cell định nghĩa biến

✅ Kiểm tra chính tả; trong notebook dùng **Restart & Run All** để chắc chắn thứ tự chạy đúng (xem [bài 00 · 02](../00_course_orientation/02_jupyter_notebook_and_markdown.ipynb) về *hidden state*).

### 3.2 `TypeError` — sai kiểu dữ liệu hoặc sai cách gọi hàm

In [ ]:
show_error("age = 20\nprint('Tuổi: ' + age)")                      # cộng str với int
show_error("def area(w, h):\n    return w * h\narea(3)")             # thiếu tham số
show_error("def scale(xs):\n    [x * 2 for x in xs]\nscale([1, 2])[0]")   # quên return → None
show_error("counts = {}\ncounts[[1, 2]] = 3")                       # list không làm key được

| Thông điệp | Sửa |
|---|---|
| `can only concatenate str (not "int") to str` | dùng f-string: `f"Tuổi: {age}"` |
| `missing 1 required positional argument` | truyền đủ tham số — đọc lại chữ ký hàm |
| `'NoneType' object is not subscriptable` | một hàm **không `return`** (trả về `None`) — thêm `return` |
| `unhashable type: 'list'` | key của dict/phần tử của set phải bất biến → dùng `tuple` |

### 3.3 `ValueError` — đúng kiểu nhưng giá trị không hợp lệ

In [ ]:
show_error("int('3.5')")                         # chuỗi không phải số nguyên
show_error("a, b = [1, 2, 3]")                    # số biến ≠ số phần tử khi unpack
show_error("math.sqrt(-1)")                       # ngoài miền xác định
show_error("np.array([1, 2, 3]).reshape(2, 2)")   # 3 phần tử không thể thành 2×2

✅ `int(float("3.5"))` nếu thật sự muốn làm tròn; kiểm tra độ dài trước khi unpack; với `reshape`, tích các chiều phải bằng số phần tử.

### 3.4 `IndexError` và `KeyError` — truy cập phần tử không tồn tại

In [ ]:
show_error("scores = [0.8, 0.9, 0.7]\nfor i in range(len(scores) + 1):\n    print(scores[i])")   # lỗi lệch một (off-by-one)
show_error("config = {'lr': 0.01}\nprint(config['learning_rate'])")                             # sai key

In [ ]:
config = {"lr": 0.01}
print(config.get("epochs", 10))       # ✅ có giá trị mặc định
print("epochs" in config)              # ✅ kiểm tra trước khi lấy
for i, s in enumerate([0.8, 0.9, 0.7]):  # ✅ duyệt trực tiếp, không tự đếm chỉ số
    print(i, s)

> 🔎 Với pandas, `KeyError: 'price '` thường do tên cột có **khoảng trắng thừa** hoặc sai hoa/thường — in `df.columns.tolist()` để kiểm tra.

### 3.5 `AttributeError` — object không có thuộc tính/phương thức đó

In [ ]:
show_error("data = [1, 2, 3]\nprint(data.shape)")              # list không phải ndarray
show_error("xs = [3, 1, 2]\nxs = xs.sort()\nxs.append(4)")     # .sort() sửa tại chỗ và trả về None

✅ `np.asarray(data).shape`; dùng `xs.sort()` (không gán lại) hoặc `xs = sorted(xs)`. Quy tắc: phương thức **sửa tại chỗ** (`sort`, `append`, `extend`, `reverse`) trả về `None`.

### 3.6 `ZeroDivisionError` và `OverflowError`

In [ ]:
show_error("print(10 / 0)")
show_error("print(math.exp(1000))")
with np.errstate(over="ignore", divide="ignore"):
    print("NumPy không báo lỗi mà trả về:", np.float64(1.0) / 0, np.exp(1000.0))

Python thuần **dừng** với lỗi; NumPy chỉ **cảnh báo** và trả về `inf`/`nan` — nguy hiểm hơn vì code vẫn chạy tiếp (xem [bài 06](06_debugging_shapes_and_performance.ipynb), mục NaN/inf).

### 3.7 `ModuleNotFoundError` / `ImportError` — chưa cài thư viện hoặc sai môi trường

In [ ]:
show_error("import sklearn_not_installed")
show_error("from math import sqroot")

| Tình huống | Sửa |
|---|---|
| Đã `pip install` mà notebook vẫn báo lỗi | kernel đang dùng **môi trường khác** → cài bằng `%pip install tên-gói` ngay trong notebook, hoặc chọn đúng kernel |
| Tên để `import` khác tên để cài | `sklearn` ↔ `scikit-learn`, `cv2` ↔ `opencv-python`, `PIL` ↔ `pillow`, `yaml` ↔ `pyyaml` |
| `cannot import name …` | sai tên hàm, hoặc phiên bản thư viện khác — kiểm tra `thư_viện.__version__` |

### 3.8 `FileNotFoundError` — đường dẫn tương đối phụ thuộc thư mục làm việc

In [ ]:
import os
from pathlib import Path

show_error("open('data/sample/house_prices.csv')")
print("Thư mục làm việc hiện tại:", Path(os.getcwd()).name)
print("Đường dẫn đúng từ notebook của chương:", Path("../../data/sample/house_prices.csv").exists())

✅ Notebook chạy với thư mục làm việc = thư mục chứa notebook. Dùng đường dẫn tương đối **từ notebook** (`../../data/...`), hoặc `repo_path("data", "sample", ...)` của khoá học (không phụ thuộc thư mục làm việc), và `pathlib.Path` thay vì ghép chuỗi.

### 3.9 `UnboundLocalError` — gán vào biến toàn cục bên trong hàm

In [ ]:
show_error("n_calls = 0\ndef predict(x):\n    n_calls += 1\n    return x\npredict(1)")

Có phép gán `n_calls += 1` trong hàm nên Python coi `n_calls` là biến **cục bộ** — nhưng nó chưa có giá trị. ✅ Truyền vào/trả ra giá trị, hoặc dùng class giữ trạng thái; tránh `global`.

### 3.10 `KeyboardInterrupt`, `MemoryError` và kernel "chết"

- Cell chạy mãi không dừng → **Interrupt** (■) gửi `KeyboardInterrupt`; kiểm tra vòng lặp `while` có điều kiện dừng không.
- `MemoryError` hoặc *"The kernel appears to have died"* thường do hết RAM: ma trận quá lớn (vd ma trận khoảng cách $n \times n$ với $n = 10^5$ cần 80 GB), giữ nhiều bản sao dữ liệu. ✅ Ước lượng bộ nhớ trước: `n * d * 8` byte với `float64`; dùng `float32`, xử lý theo lô (batch).

In [ ]:
n = 100_000
print(f"Ma trận khoảng cách {n:,}×{n:,} float64 cần {n * n * 8 / 1e9:,.0f} GB RAM")

## 4. Lỗi logic — không có exception, kết quả vẫn sai

Nguy hiểm nhất vì **không có traceback** nào cảnh báo.

In [ ]:
a, b = [1, 2], [1, 2]
print("1) a == b:", a == b, "| a is b:", a is b)          # == so sánh giá trị; is so sánh danh tính object
print("2) 0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3, "| isclose:", math.isclose(0.1 + 0.2, 0.3))
print("3) 7 / 2 =", 7 / 2, "| 7 // 2 =", 7 // 2, "| -7 // 2 =", -7 // 2)   # // làm tròn XUỐNG

xs = [1, 2, 2, 3]
for x in xs:            # 4) sửa list khi đang duyệt → bỏ sót phần tử
    if x == 2:
        xs.remove(x)
print("4) xóa mọi số 2 (sai):", xs, "| đúng:", [x for x in [1, 2, 2, 3] if x != 2])


def mean_bad(values):
    total = 0
    for v in values:
        total += v
        return total / len(values)   # 5) return thụt lề vào trong vòng lặp → trả về sau phần tử đầu


print("5) mean_bad([2, 4, 6]) =", mean_bad([2, 4, 6]), "| đúng:", sum([2, 4, 6]) / 3)

| Lỗi logic | Cách phòng |
|---|---|
| `is` thay cho `==` | chỉ dùng `is` với `None`, `True`, `False` |
| so sánh số thực bằng `==` | `math.isclose`, `np.isclose`, `np.allclose` |
| nhầm `/` và `//` | `/` luôn ra số thực; `//` làm tròn xuống (kể cả số âm) |
| sửa list khi đang duyệt | tạo list mới bằng comprehension |
| `return` sai mức thụt lề | viết test nhỏ với kết quả biết trước |
| aliasing, mutable default argument | xem [bài 01](01_python_basics.ipynb), mục 5 |

## 5. Xử lý lỗi đúng cách

### 5.1 `try/except` — bắt **đúng** loại lỗi, ở **đúng** chỗ

Ví dụ thực tế: cột số tiền đọc từ file có vài giá trị hỏng. Ta muốn chuyển đổi, **đếm** giá trị hỏng — không phải giấu chúng đi.

In [ ]:
raw = ["120000", "85000", "N/A", "99,000", "150000", ""]


def parse_amounts(values):
    parsed, bad = [], []
    for v in values:
        try:
            parsed.append(float(v))
        except ValueError:            # ✅ chỉ bắt lỗi ta dự liệu được
            bad.append(v)
            parsed.append(float("nan"))
    return parsed, bad


parsed, bad = parse_amounts(raw)
print("đã chuyển:", parsed)
print(f"{len(bad)} giá trị hỏng cần xem lại:", bad)

❌ **Đừng** viết `except: pass` hay `except Exception: pass` — mọi lỗi (kể cả lỗi gõ sai tên biến) đều bị nuốt im lặng:

In [ ]:
def risky_mean(values):
    try:
        return sum(valeus) / len(values)   # 🐛 gõ sai tên biến  # noqa: F821
    except Exception:
        return 0.0                          # ❌ giấu luôn cả lỗi NameError


print("risky_mean([1, 2, 3]) =", risky_mean([1, 2, 3]), "← sai mà không ai biết")

### 5.2 `raise` — báo lỗi sớm, thông điệp rõ ràng

Hàm của bạn nên **kiểm tra đầu vào** và báo lỗi ngay tại chỗ, kèm thông tin để sửa — thay vì để lỗi khó hiểu xảy ra sâu bên trong:

In [ ]:
def train_test_ratio_split(X, test_size):
    if not 0 < test_size < 1:
        raise ValueError(f"test_size phải nằm trong (0, 1), nhận được {test_size!r}")
    if len(X) < 2:
        raise ValueError(f"cần ít nhất 2 mẫu để chia, nhận được {len(X)}")
    n_test = max(1, round(len(X) * test_size))
    return X[:-n_test], X[-n_test:]


show_error("train_test_ratio_split(list(range(10)), 20)")
print(train_test_ratio_split(list(range(10)), 0.2))

### 5.3 `assert` cho giả định nội bộ, `warnings` cho vấn đề "mềm"

- `assert điều_kiện, "thông điệp"`: kiểm tra giả định **của chính code bạn** (shape, không có NaN…). Python chạy với cờ `-O` sẽ **bỏ qua** `assert`, nên **không** dùng `assert` để kiểm tra dữ liệu người dùng — dùng `raise`.
- `warnings.warn(...)`: báo điều đáng chú ý nhưng không cần dừng (vd dữ liệu có ít mẫu).

In [ ]:
def check_features(X):
    X = np.asarray(X, dtype=float)
    assert X.ndim == 2, f"X phải là ma trận 2 chiều, nhận shape {X.shape}"
    if len(X) < 30:
        warnings.warn(f"chỉ có {len(X)} mẫu — kết quả đánh giá sẽ rất dao động", stacklevel=2)
    return X


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    check_features(np.ones((10, 3)))
print("cảnh báo:", caught[0].message)
show_error("assert np.ones(5).ndim == 2, 'X phải là ma trận 2 chiều'")

## 6. Quy trình gỡ lỗi 5 bước

1. **Đọc dòng cuối** traceback: loại lỗi + thông điệp.
2. **Tìm dòng code của bạn** gần nhất với lỗi.
3. **Tái hiện tối giản**: rút về vài dòng vẫn gây lỗi — thường tự bạn sẽ thấy nguyên nhân (đây cũng là cách đặt câu hỏi tốt, xem [bài 00 · 05](../00_course_orientation/05_google_search_and_learning_skills.ipynb)).
4. **Quan sát giá trị**: `print`/`describe_array` trước dòng lỗi; trong Jupyter, chạy `%debug` ngay sau khi lỗi để vào trình gỡ lỗi tại đúng chỗ đó (`p tên_biến` để in, `u`/`d` để lên/xuống lời gọi, `q` để thoát); trong file `.py` dùng `breakpoint()`.
5. **Sửa và chặn tái phát**: thêm một `assert`/test nhỏ để lỗi không quay lại.

## 7. Tự luyện: đoán lỗi trước khi chạy

Với mỗi đoạn code, **đoán** loại lỗi (hoặc "không lỗi nhưng sai") rồi mới chạy cell kiểm tra.

In [ ]:
puzzles = {
    "A": "prices = {'hn': 3.2, 'hcm': 4.1}\nprint(prices['HCM'])",
    "B": "values = [3, 1, 2]\nbest = values.sort()\nprint(best[0])",
    "C": "n = '5'\nprint(n * 2)",
    "D": "w = np.zeros(3)\nX = np.ones((4, 2))\nprint(X @ w)",
    "E": "import math\nprint(math.log(0))",
}
for name, code in puzzles.items():
    print(f"[{name}]", end=" ")
    show_error(code)

<details><summary>💬 Giải thích</summary>

- **A** — `KeyError: 'HCM'`: key phân biệt hoa/thường.
- **B** — `TypeError: 'NoneType' object is not subscriptable`: `.sort()` trả về `None`.
- **C** — **không lỗi nhưng có thể sai ý**: `'5' * 2 == '55'` (nhân chuỗi), không phải `10` → ép kiểu `int(n)`.
- **D** — `ValueError` của `matmul`: `X` có 2 cột nhưng `w` có 3 phần tử — lỗi shape kinh điển (bài 06).
- **E** — `ValueError` (*math domain error*; Python ≥ 3.14 ghi rõ hơn là cần số dương): $\log 0$ không xác định; với NumPy thì ra `-inf` kèm cảnh báo.

</details>

## 🧠 Kiểm tra nhanh

1. Khi đọc traceback, bạn nhìn dòng nào đầu tiên? Vì sao?
2. Hàm `f` của bạn không có `return`. `f(x)[0]` báo lỗi gì?
3. Vì sao `except Exception: pass` nguy hiểm?
4. Khi nào dùng `raise ValueError(...)`, khi nào dùng `assert`?
5. Đã `pip install lightgbm` trong terminal nhưng notebook vẫn `ModuleNotFoundError`. Nguyên nhân khả dĩ nhất và cách sửa?

## 📌 Tóm tắt

- Đọc traceback **từ dưới lên**; tìm dòng đầu tiên thuộc code của bạn.
- Lỗi cú pháp: thiếu `:`, thiếu ngoặc, `=` thay `==`, thụt lề. Lỗi runtime hay gặp: `NameError`, `TypeError`, `ValueError`, `IndexError`, `KeyError`, `AttributeError`, `ZeroDivisionError`, `ModuleNotFoundError`, `FileNotFoundError`, `UnboundLocalError`.
- Lỗi logic không có exception — phòng bằng test nhỏ với kết quả biết trước.
- `try/except` bắt **đúng loại** lỗi; `raise` với thông điệp rõ ràng cho dữ liệu đầu vào; `assert` cho giả định nội bộ.

## 📚 Tài liệu tham khảo

- Python Tutorial — [Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html); danh sách [Built-in Exceptions](https://docs.python.org/3/library/exceptions.html).
- AIVIETNAM — *Những lỗi thường gặp trong Python* (bài đọc tuần 1 của khoá AI Summer 2019).
- Real Python — [Understanding the Python Traceback](https://realpython.com/python-traceback/).

<sub>Chủ đề theo kế hoạch tuần 1 của khoá [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning) (bản gốc chỉ có liên kết ngoài); nội dung soạn mới cho khoá học này.</sub>